# Kickstarter Regression Preprocessing — Final Encoding

This notebook starts from:

`kickstarter_clean_before_encoding.csv`

and creates exactly **two final files**:

- `kickstarter_train_final.csv`
- `kickstarter_test_final.csv`

Each file contains:

- all encoded model features
- `target_usd`
- `log_target`

There are no separate label files and no model training.

Important:

- Split is 80% train / 20% test
- Frequency mappings are learned from training data only
- One-hot encoder is fitted on training data only
- Numerical medians are calculated from training data only


## Cell 1 — Imports


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)


## Cell 2 — Load cleaned dataset


In [2]:
DATA_FILE = r"E:\NSU\cse445\EDA attempt3\Dataset\kickstarter_raw_before_encoding.csv"

df = pd.read_csv(
    DATA_FILE,
    keep_default_na=False
)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (20000, 25)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,launch_month,launch_day,launch_weekday,launch_hour,has_video,prelaunch_activated,country,currency,category_parent,category_name,location_type,location_country,location_state,target_usd,log_target
0,800050031,6000.000000,8.699681,29.958333,8.745764,33,4,128,18,2013,2,8,4,12,1,0,US,USD,Music,Latin,Town,US,KY,10043.000000,9.214731
1,746075806,11655.899250,9.363653,30.416447,116.766181,12,1,104,19,2018,5,22,1,14,1,0,CA,CAD,Film & Video,Documentary,Town,CA,ON,36295.918552,10.499488
2,1638770732,106.553578,4.677989,27.000000,4.452442,50,7,104,22,2022,12,31,5,23,1,1,DE,EUR,Art,Painting,LocalAdmin,DE,Rhineland-Palatinate,3325.537169,8.109687
3,511984273,20000.000000,9.903538,30.000000,14.753009,13,3,62,8,2023,4,21,4,15,1,1,US,USD,Film & Video,Horror,Town,US,AL,30020.660000,10.309674
4,752979518,477.029064,6.169672,30.603519,27.764248,44,8,99,14,2022,3,24,3,13,0,0,CA,CAD,Fashion,Accessories,Town,CA,ON,736.445419,6.603192


## Cell 3 — Define columns


In [3]:
ID_COL = "id"

TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"

categorical_columns = [
    "country",
    "currency",
    "category_parent",
    "category_name",
    "location_type",
    "location_country",
    "location_state",
]

print("Categorical columns:")
for col in categorical_columns:
    print(col)


Categorical columns:
country
currency
category_parent
category_name
location_type
location_country
location_state


## Cell 4 — Clean categorical values


In [4]:
for col in categorical_columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown",
            "nan": "Unknown",
            "None": "Unknown"
        })
    )

print("Categorical cleaning complete.")


Categorical cleaning complete.


## Cell 5 — Create target bins for stratified split


In [5]:
df["_target_bin"] = pd.qcut(
    df[TARGET_LOG],
    q=10,
    labels=False,
    duplicates="drop"
)

print(
    df["_target_bin"]
    .value_counts()
    .sort_index()
)


_target_bin
0    2000
1    2048
2    1952
3    2003
4    1997
5    2001
6    2000
7    1999
8    2000
9    2000
Name: count, dtype: int64


## Cell 6 — Split into 80% train and 20% test


In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["_target_bin"]
)

train_df = (
    train_df
    .drop(columns=["_target_bin"])
    .reset_index(drop=True)
)

test_df = (
    test_df
    .drop(columns=["_target_bin"])
    .reset_index(drop=True)
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))


Train rows: 16000
Test rows: 4000


## Cell 7 — Identify low- and high-cardinality categorical columns


In [7]:
CARDINALITY_THRESHOLD = 30

low_cardinality = []
high_cardinality = []

for col in categorical_columns:

    n_unique = train_df[col].nunique()

    if n_unique <= CARDINALITY_THRESHOLD:
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

print("LOW CARDINALITY → ONE-HOT")
for col in low_cardinality:
    print(
        f"{col:25s}",
        train_df[col].nunique()
    )

print("\nHIGH CARDINALITY → FREQUENCY")
for col in high_cardinality:
    print(
        f"{col:25s}",
        train_df[col].nunique()
    )


LOW CARDINALITY → ONE-HOT
country                   25
currency                  15
category_parent           15
location_type             9

HIGH CARDINALITY → FREQUENCY
category_name             160
location_country          127
location_state            520


## Cell 8 — Identify numerical features


In [8]:
excluded_columns = set(
    categorical_columns
    + [
        ID_COL,
        TARGET_RAW,
        TARGET_LOG
    ]
)

numeric_features = [
    col
    for col in train_df.columns
    if col not in excluded_columns
]

print("Numerical features:")
for col in numeric_features:
    print(col)

print(
    "\nNumber of numerical features:",
    len(numeric_features)
)


Numerical features:
goal_usd
log_goal_usd
duration_days
prelaunch_days
name_char_length
name_word_count
blurb_char_length
blurb_word_count
launch_year
launch_month
launch_day
launch_weekday
launch_hour
has_video
prelaunch_activated

Number of numerical features: 15


## Cell 9 — Convert numerical columns to numeric


In [9]:
for data in [
    train_df,
    test_df
]:

    for col in numeric_features:

        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )

        data[col] = data[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

print("Numerical conversion complete.")


Numerical conversion complete.


## Cell 10 — Calculate training medians


In [10]:
numeric_medians = (
    train_df[numeric_features]
    .median()
)

display(
    numeric_medians.to_frame(
        "training_median"
    )
)


,training_median
goal_usd,4523.886864
log_goal_usd,8.417348
duration_days,30.000000
prelaunch_days,15.054919
name_char_length,36.000000
name_word_count,6.000000
blurb_char_length,116.000000
blurb_word_count,18.000000
launch_year,2020.000000
launch_month,7.000000


## Cell 11 — Fill numerical missing values


In [11]:
train_df[numeric_features] = (
    train_df[numeric_features]
    .fillna(numeric_medians)
)

test_df[numeric_features] = (
    test_df[numeric_features]
    .fillna(numeric_medians)
)

print(
    "Train missing numeric:",
    train_df[numeric_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Test missing numeric:",
    test_df[numeric_features]
    .isna()
    .sum()
    .sum()
)


Train missing numeric: 0
Test missing numeric: 0


# Frequency Encoding


## Cell 12 — Build frequency maps from training data only


In [12]:
frequency_maps = {}

for col in high_cardinality:

    frequency_maps[col] = (
        train_df[col]
        .value_counts(
            normalize=True
        )
        .to_dict()
    )

print("Frequency maps created for:")
print(high_cardinality)


Frequency maps created for:
['category_name', 'location_country', 'location_state']


## Cell 13 — Frequency encoding function


In [13]:
def apply_frequency_encoding(
    data,
    columns,
    frequency_maps
):

    result = pd.DataFrame(
        index=data.index
    )

    for col in columns:

        new_col = col + "_freq"

        result[new_col] = (
            data[col]
            .map(frequency_maps[col])
            .fillna(0)
            .astype(np.float32)
        )

    return result


## Cell 14 — Apply frequency encoding


In [14]:
train_freq = apply_frequency_encoding(
    train_df,
    high_cardinality,
    frequency_maps
)

test_freq = apply_frequency_encoding(
    test_df,
    high_cardinality,
    frequency_maps
)

print("Frequency columns:")
print(train_freq.columns.tolist())

display(train_freq.head())


Frequency columns:
['category_name_freq', 'location_country_freq', 'location_state_freq']


,category_name_freq,location_country_freq,location_state_freq
0,0.006625,0.045938,0.009750
1,0.009813,0.622312,0.037000
2,0.006125,0.622312,0.106500
3,0.015812,0.003125,0.001500
4,0.011188,0.622312,0.033125


# One-Hot Encoding


## Cell 15 — Create one-hot encoder


In [15]:
try:

    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False,
        dtype=np.float32
    )

except TypeError:

    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse=False,
        dtype=np.float32
    )


## Cell 16 — Fit one-hot encoder on training data only


In [16]:
onehot_encoder.fit(
    train_df[low_cardinality]
)

onehot_columns = (
    onehot_encoder
    .get_feature_names_out(
        low_cardinality
    )
)

print(
    "Number of one-hot columns:",
    len(onehot_columns)
)


Number of one-hot columns: 60


## Cell 17 — Transform train and test


In [17]:
train_ohe = pd.DataFrame(
    onehot_encoder.transform(
        train_df[low_cardinality]
    ),
    columns=onehot_columns
)

test_ohe = pd.DataFrame(
    onehot_encoder.transform(
        test_df[low_cardinality]
    ),
    columns=onehot_columns
)

print(
    "Train one-hot shape:",
    train_ohe.shape
)

print(
    "Test one-hot shape:",
    test_ohe.shape
)


Train one-hot shape: (16000, 60)
Test one-hot shape: (4000, 60)


## Cell 18 — Prepare numerical features


In [18]:
train_numeric = (
    train_df[numeric_features]
    .reset_index(drop=True)
    .astype(np.float32)
)

test_numeric = (
    test_df[numeric_features]
    .reset_index(drop=True)
    .astype(np.float32)
)

train_freq = (
    train_freq
    .reset_index(drop=True)
)

test_freq = (
    test_freq
    .reset_index(drop=True)
)


## Cell 19 — Combine all model features


In [19]:
X_train = pd.concat(
    [
        train_numeric,
        train_freq,
        train_ohe
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_numeric,
        test_freq,
        test_ohe
    ],
    axis=1
)

print(
    "X_train shape:",
    X_train.shape
)

print(
    "X_test shape:",
    X_test.shape
)


X_train shape: (16000, 78)
X_test shape: (4000, 78)


## Cell 20 — Verify feature count and consistency


In [20]:
assert list(X_train.columns) == list(X_test.columns)

assert ID_COL not in X_train.columns
assert TARGET_RAW not in X_train.columns
assert TARGET_LOG not in X_train.columns

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("All checks passed.")

print(
    "Numerical features:",
    len(numeric_features)
)

print(
    "Frequency features:",
    train_freq.shape[1]
)

print(
    "One-hot features:",
    train_ohe.shape[1]
)

print(
    "TOTAL MODEL FEATURES:",
    X_train.shape[1]
)


All checks passed.
Numerical features: 15
Frequency features: 3
One-hot features: 60
TOTAL MODEL FEATURES: 78


## Cell 21 — Show final feature names


In [21]:
for i, col in enumerate(
    X_train.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


01. goal_usd
02. log_goal_usd
03. duration_days
04. prelaunch_days
05. name_char_length
06. name_word_count
07. blurb_char_length
08. blurb_word_count
09. launch_year
10. launch_month
11. launch_day
12. launch_weekday
13. launch_hour
14. has_video
15. prelaunch_activated
16. category_name_freq
17. location_country_freq
18. location_state_freq
19. country_AU
20. country_BE
21. country_CA
22. country_CH
23. country_DE
24. country_DK
25. country_ES
26. country_FR
27. country_GB
28. country_GR
29. country_HK
30. country_IE
31. country_IT
32. country_JP
33. country_LU
34. country_MX
35. country_NL
36. country_NO
37. country_NZ
38. country_PL
39. country_SE
40. country_SG
41. country_SI
42. country_US
43. currency_CAD
44. currency_CHF
45. currency_DKK
46. currency_EUR
47. currency_GBP
48. currency_HKD
49. currency_JPY
50. currency_MXN
51. currency_NOK
52. currency_NZD
53. currency_PLN
54. currency_SEK
55. currency_SGD
56. currency_USD
57. category_parent_Comics
58. category_parent_Crafts
59.

## Cell 22 — Create a single final training file

The final training CSV contains:

- ID
- encoded features
- `target_usd`
- `log_target`

When you later train a model, remove `id`, `target_usd`, and `log_target` from `X`.


In [22]:
train_final = pd.concat(
    [
        train_df[[ID_COL]]
        .reset_index(drop=True),

        X_train,

        train_df[
            [
                TARGET_RAW,
                TARGET_LOG
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

print(
    "Final training shape:",
    train_final.shape
)

display(train_final.head())


Final training shape: (16000, 81)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,launch_month,launch_day,launch_weekday,launch_hour,has_video,prelaunch_activated,category_name_freq,location_country_freq,location_state_freq,country_AU,country_BE,country_CA,country_CH,country_DE,country_DK,country_ES,country_FR,country_GB,country_GR,country_HK,country_IE,country_IT,country_JP,country_LU,country_MX,country_NL,country_NO,country_NZ,country_PL,country_SE,country_SG,country_SI,country_US,currency_CAD,currency_CHF,currency_DKK,currency_EUR,currency_GBP,currency_HKD,currency_JPY,currency_MXN,currency_NOK,currency_NZD,currency_PLN,currency_SEK,currency_SGD,currency_USD,category_parent_Comics,category_parent_Crafts,category_parent_Dance,category_parent_Design,category_parent_Fashion,category_parent_Film & Video,category_parent_Food,category_parent_Games,category_parent_Journalism,category_parent_Music,category_parent_Photography,category_parent_Publishing,category_parent_Technology,category_parent_Theater,location_type_County,location_type_Island,location_type_LocalAdmin,location_type_Miscellaneous,location_type_Suburb,location_type_Town,location_type_Unknown,location_type_Zip,target_usd,log_target
0,160053502,1490.124146,7.307286,30.000000,13.537003,30.0,5.0,25.0,5.0,2020.0,6.0,8.0,0.0,15.0,1.0,1.0,0.006625,0.045938,0.009750,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2694.360622,7.899287
1,1797462698,5000.000000,8.517393,35.041668,18.665070,8.0,2.0,108.0,18.0,2017.0,10.0,17.0,1.0,13.0,1.0,0.0,0.009813,0.622312,0.037000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,122.000000,4.812184
2,1582340481,10000.000000,9.210441,30.000000,10.058496,53.0,7.0,108.0,13.0,2025.0,1.0,3.0,4.0,2.0,1.0,0.0,0.006125,0.622312,0.106500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.000000
3,498571554,15475.735352,9.647093,29.958334,7.044641,39.0,7.0,110.0,18.0,2022.0,3.0,11.0,4.0,12.0,0.0,1.0,0.015812,0.003125,0.001500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,26102.038065,10.169807
4,1914908476,1400.000000,7.244942,60.000000,13.044236,29.0,5.0,125.0,24.0,2025.0,3.0,23.0,6.0,21.0,1.0,0.0,0.011188,0.622312,0.033125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1478.000000,7.299121


## Cell 23 — Create a single final test file


In [23]:
test_final = pd.concat(
    [
        test_df[[ID_COL]]
        .reset_index(drop=True),

        X_test,

        test_df[
            [
                TARGET_RAW,
                TARGET_LOG
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

print(
    "Final test shape:",
    test_final.shape
)

display(test_final.head())


Final test shape: (4000, 81)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,launch_month,launch_day,launch_weekday,launch_hour,has_video,prelaunch_activated,category_name_freq,location_country_freq,location_state_freq,country_AU,country_BE,country_CA,country_CH,country_DE,country_DK,country_ES,country_FR,country_GB,country_GR,country_HK,country_IE,country_IT,country_JP,country_LU,country_MX,country_NL,country_NO,country_NZ,country_PL,country_SE,country_SG,country_SI,country_US,currency_CAD,currency_CHF,currency_DKK,currency_EUR,currency_GBP,currency_HKD,currency_JPY,currency_MXN,currency_NOK,currency_NZD,currency_PLN,currency_SEK,currency_SGD,currency_USD,category_parent_Comics,category_parent_Crafts,category_parent_Dance,category_parent_Design,category_parent_Fashion,category_parent_Film & Video,category_parent_Food,category_parent_Games,category_parent_Journalism,category_parent_Music,category_parent_Photography,category_parent_Publishing,category_parent_Technology,category_parent_Theater,location_type_County,location_type_Island,location_type_LocalAdmin,location_type_Miscellaneous,location_type_Suburb,location_type_Town,location_type_Unknown,location_type_Zip,target_usd,log_target
0,1230232537,815.564453,6.705106,33.060844,3.941562,38.0,7.0,27.0,7.0,2024.0,5.0,7.0,1.0,7.0,0.0,0.0,0.014625,0.020063,0.004125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4212.076879,8.345949
1,633033535,2000.000000,7.601402,31.000000,0.267234,60.0,12.0,134.0,22.0,2011.0,5.0,23.0,0.0,17.0,1.0,0.0,0.009375,0.000375,0.000250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2000.000000,7.601402
2,455340929,1644.969116,7.406085,21.099236,30.939156,41.0,6.0,108.0,19.0,2021.0,1.0,30.0,5.0,19.0,1.0,1.0,0.008438,0.109438,0.009500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,6922.578178,8.842688
3,775293744,10000.000000,9.210441,59.958332,0.051539,18.0,2.0,133.0,20.0,2016.0,3.0,9.0,2.0,16.0,0.0,0.0,0.010812,0.622312,0.106500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.000000,0.693147
4,1437303150,13913.663086,9.540698,30.000000,23.272894,56.0,9.0,123.0,22.0,2014.0,5.0,9.0,4.0,19.0,1.0,0.0,0.013813,0.006750,0.002750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,313.057415,5.749576


## Cell 24 — Final verification


In [24]:
assert list(
    train_final.columns[1:-2]
) == list(
    test_final.columns[1:-2]
)

assert train_final.isna().sum().sum() == 0
assert test_final.isna().sum().sum() == 0

print("Final train/test files verified.")

print(
    "\nTrain rows:",
    len(train_final)
)

print(
    "Test rows:",
    len(test_final)
)

print(
    "Model input features:",
    X_train.shape[1]
)

print(
    "Total columns in each saved file:",
    train_final.shape[1]
)


Final train/test files verified.

Train rows: 16000
Test rows: 4000
Model input features: 78
Total columns in each saved file: 81


## Cell 25 — Save exactly two final CSV files


In [25]:
train_final.to_csv(
    "kickstarter_train_final.csv",
    index=False
)

test_final.to_csv(
    "kickstarter_test_final.csv",
    index=False
)

print("Saved:")
print("kickstarter_train_final.csv")
print("kickstarter_test_final.csv")


Saved:
kickstarter_train_final.csv
kickstarter_test_final.csv


## Final Output

Only two final datasets are produced:

### `kickstarter_train_final.csv`

Contains:

- `id`
- encoded model features
- `target_usd`
- `log_target`

### `kickstarter_test_final.csv`

Contains the same structure.

No model is trained in this notebook.

Later:

```python
X_train = train_df.drop(
    columns=["id", "target_usd", "log_target"]
)

y_train = train_df["log_target"]
```

and similarly for the test set.
